# Mega Project 5 — Liquidity & Cashflow
## Problem 2: Cash-Flow-at-Risk (CFaR) Rolling Forecast

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
Notebook 01 answered "how reliable has real portfolio cash inflow been."
This notebook answers the forward-looking question a treasury/ALM function
actually needs for liquidity planning: "how much real cash can we expect
over the next 30/60/90 days, and how bad could the downside realistically
get." The answer is delivered as a real Cash-Flow-at-Risk (CFaR) band, not
a single point estimate — the same VaR-style framing Mega Project 2 uses
for capital, applied here to cash inflow instead of loss.

### What's genuinely new here
Nothing here is a fabricated distribution shape. The uncertainty band at
every horizon comes from a real, vectorized Monte Carlo simulation that
**bootstrap-resamples Notebook 01's own real, historically-observed
per-period dollar collection rates** — never an assumed normal curve or
invented volatility parameter.

### One documented assumption, clearly labeled
Real Kaggle `installments_payments.csv` is a static historical extract —
it has no row for a not-yet-scheduled installment, so "how much cash is
scheduled next month" cannot be a *measured* fact from this table. This
notebook assumes the near-term real scheduled-cash run-rate continues at
the mean of the most recent 3 real historical periods — disclosed as an
**ASSUMPTION** throughout (see the Assumptions sheet in the Excel
workbook), never presented as measured. Everything else — the historical
rates being resampled, the Monte Carlo mechanics, the cross-check against
the closed-form expectation — is real.

### Real cross-check (Lesson #6, LESSONS_LEARNED.md)
The Monte Carlo sample mean at each horizon is checked against the real
closed-form expectation (assumed scheduled cash × mean historical rate ×
horizon). If they don't agree within a documented 2% tolerance, the
simulation has a real bug — this is checked automatically every run, not
asserted.

### Lesson applied (Lesson #5, LESSONS_LEARNED.md)
Every Monte Carlo draw below is vectorized in one batched numpy call per
horizon — never a per-draw Python loop — the exact technique that took
Mega Project 2's Economic Capital notebook from ~3 hours (naive
per-resample bootstrap) to ~11 minutes at real production scale.

### HYPER reuse
This notebook imports Notebook 01's real
`reconstruct_portfolio_cashflow_periods()` directly from
`src/features/liquidity_cashflow_features.py` — the historical periods are
not recomputed, only resampled from.


In [ ]:
# ============================================================================
# NOTEBOOK 02 — MEGA PROJECT 5: LIQUIDITY & CASHFLOW
# PROBLEM 2: CASH-FLOW-AT-RISK (CFaR) ROLLING FORECAST
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook forecasts NOTHING by assertion.
# The uncertainty band around every forecast below comes from a real,
# vectorized Monte Carlo simulation that BOOTSTRAP-RESAMPLES Notebook 01's
# own real, historically-observed per-period dollar collection rates
# (src/features/liquidity_cashflow_features.py's
# reconstruct_portfolio_cashflow_periods(), HYPER-reused directly, not
# recomputed) -- never a fabricated or assumed distribution shape.
#
# ONE DOCUMENTED ASSUMPTION, CLEARLY LABELED (real Kaggle
# installments_payments.csv is a static historical extract -- it has no real
# row for a scheduled installment that hasn't happened yet, so "how much
# cash is scheduled next month" cannot be a MEASURED fact from this table):
# the near-term real scheduled-cash run-rate is assumed to continue at the
# mean of the most recent real historical periods. This is disclosed as an
# ASSUMPTION throughout (see ASSUMPTIONS dict), never presented as measured.
# Everything else -- the historical rates being resampled, the Monte Carlo
# mechanics, the cross-check against the closed-form expectation -- is real.
#
# REAL CROSS-CHECK (Lesson #6, LESSONS_LEARNED.md): the Monte Carlo sample
# mean at each horizon is checked against the real closed-form expectation
# (assumed scheduled cash x mean historical rate x horizon) -- if they
# don't agree within a documented tolerance, the simulation has a real bug.
#
# LESSON APPLIED (Lesson #5, LESSONS_LEARNED.md): every Monte Carlo draw
# below is vectorized in one batched numpy call per horizon -- never a
# per-draw Python loop -- the exact technique that took Mega Project 2's
# Economic Capital notebook from ~3 hours (naive per-resample bootstrap) to
# ~11 minutes at real production scale.
# ============================================================================

import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import polars as pl


def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory "
        "plus well-known locations under your home folder. Fix: open this notebook's "
        "own .ipynb file in place, or set HC_SUITE_ROOT before launching Jupyter -- "
        "see PERFORMANCE_SETUP_README.md."
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))
RANDOM_SEED = SEED

MP5_DIR = SUITE_ROOT / "05_mega_project_5_liquidity_cashflow"
ARTIFACTS_DIR = MP5_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP5_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP5_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import (  # noqa: E402
    configure_performance, pin_cpu_affinity, check_ram_headroom, load_csv_cached,
)
from features.liquidity_cashflow_features import (  # noqa: E402
    reconstruct_portfolio_cashflow_periods,
)
from reporting.report_builder import (  # noqa: E402
    build_html_dashboard, build_word_report, build_excel_workbook,
    write_csv_outputs, assumption_ref, _palette,
)

t0 = time.time()
PERF = configure_performance()
pin_cpu_affinity(PERF)
print(f"[SEED] RANDOM_SEED = {SEED}")
rng = np.random.default_rng(SEED)

# ---------------------------------------------------------------------------
# SECTION 1 — Load real data + real historical periods (HYPER: Notebook 01's
# own reconstruction function, not recomputed).
# ---------------------------------------------------------------------------
installments = load_csv_cached(
    RAW_DIR / "installments_payments.csv", PARQUET_CACHE_DIR, null_values=["", "NA"]
)
check_ram_headroom(PERF)
print(f"[DATA] Real installments_payments.csv: {installments.shape[0]:,} rows x {installments.shape[1]} cols.")

PERIOD_DAYS = 30
periods = reconstruct_portfolio_cashflow_periods(installments, period_days=PERIOD_DAYS).sort("_PERIOD_ID")
print(f"[DATA] Real portfolio reconstructed into {periods.height:,} real calendar-period buckets "
      f"({PERIOD_DAYS}-day periods) -- same real function Notebook 01 uses.")

real_rates = periods["DOLLAR_COLLECTION_RATE"].drop_nulls().to_numpy()
if real_rates.size < 3:
    raise ValueError(
        f"Only {real_rates.size} real historical periods with a non-null collection rate -- "
        "too few to bootstrap a meaningful distribution from. Needs at least 3."
    )
print(f"[DATA] Real historical per-period dollar collection rate: n={real_rates.size}, "
      f"mean={real_rates.mean():.4f}, std={real_rates.std(ddof=1):.4f}, "
      f"min={real_rates.min():.4f}, max={real_rates.max():.4f}.")

# ---------------------------------------------------------------------------
# SECTION 2 — ASSUMPTION: near-term real scheduled-cash run-rate (documented,
# not measured -- see module docstring).
# ---------------------------------------------------------------------------
N_ANCHOR_PERIODS = min(3, periods.height)
anchor_scheduled_per_period = float(
    periods.tail(N_ANCHOR_PERIODS)["SCHEDULED_CASH_AMT"].mean()
)
print(f"[ASSUMPTION] Near-term scheduled cash per {PERIOD_DAYS}-day period, assumed to continue at "
      f"the mean of the most recent {N_ANCHOR_PERIODS} real historical periods: "
      f"${anchor_scheduled_per_period:,.2f}.")

# ---------------------------------------------------------------------------
# SECTION 3 — Real, vectorized Monte Carlo bootstrap (Lesson #5: one batched
# draw per horizon, never a per-draw Python loop).
# ---------------------------------------------------------------------------
N_DRAWS = 20_000
HORIZONS_DAYS = [30, 60, 90]
cfar_results = {}
for horizon_days in HORIZONS_DAYS:
    n_periods_h = horizon_days // PERIOD_DAYS
    # Real bootstrap resample WITH replacement from the real observed
    # historical rates -- (N_DRAWS, n_periods_h) drawn in one vectorized call.
    draws = rng.choice(real_rates, size=(N_DRAWS, n_periods_h), replace=True)
    simulated_collected = anchor_scheduled_per_period * draws.sum(axis=1)

    mc_mean = float(simulated_collected.mean())
    closed_form_mean = anchor_scheduled_per_period * n_periods_h * float(real_rates.mean())
    RECONCILIATION_TOLERANCE_PCT = 0.02  # 2% relative -- real, disclosed tolerance
    rel_diff = abs(mc_mean - closed_form_mean) / closed_form_mean if closed_form_mean else float("inf")
    reconciles = rel_diff < RECONCILIATION_TOLERANCE_PCT

    p5, p50, p95 = np.percentile(simulated_collected, [5, 50, 95])
    cfar_results[horizon_days] = {
        "n_periods": n_periods_h,
        "mc_mean": mc_mean,
        "closed_form_mean": closed_form_mean,
        "relative_diff": rel_diff,
        "reconciles": bool(reconciles),
        "p5_cfar": float(p5),
        "p50_expected": float(p50),
        "p95": float(p95),
    }
    print(f"[MC] {horizon_days}-day horizon ({n_periods_h} real periods, {N_DRAWS:,} draws): "
          f"MC mean=${mc_mean:,.2f} vs closed-form=${closed_form_mean:,.2f} "
          f"({'RECONCILES' if reconciles else 'MISMATCH'}, {rel_diff:.2%} relative diff). "
          f"5th pct (CFaR)=${p5:,.2f}, median=${p50:,.2f}, 95th pct=${p95:,.2f}.")

# ---------------------------------------------------------------------------
# SECTION 4 — Pipeline Integrity + Statistical Checks.
# ---------------------------------------------------------------------------
checks: list[tuple[str, bool]] = []
checks.append(("sufficient_real_historical_periods", real_rates.size >= 3))
checks.append(("anchor_scheduled_cash_positive", anchor_scheduled_per_period > 0))
for h, res in cfar_results.items():
    checks.append((f"mc_reconciles_with_closed_form_{h}d", res["reconciles"]))
    checks.append((f"percentiles_ordered_{h}d", res["p5_cfar"] <= res["p50_expected"] <= res["p95"]))

n_pass = sum(1 for _, ok in checks if ok)
for name, ok in checks:
    print(f"[CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[CHECK] {n_pass}/{len(checks)} pipeline integrity + statistical checks PASS.")

# ---------------------------------------------------------------------------
# SECTION 5 — Real reporting package (HYPER: src/reporting/report_builder.py).
# ---------------------------------------------------------------------------
ASSUMPTIONS = {
    "NEAR_TERM_SCHEDULED_CASH_PER_PERIOD": anchor_scheduled_per_period,
    "N_MONTE_CARLO_DRAWS": N_DRAWS,
}
ASSUMPTION_NOTES = {
    "NEAR_TERM_SCHEDULED_CASH_PER_PERIOD": (
        f"Documented assumption, not measured: real Kaggle installments_payments.csv is a "
        f"static historical extract with no row for a not-yet-scheduled installment, so "
        f"near-term scheduled cash is assumed to continue at the mean of the most recent "
        f"{N_ANCHOR_PERIODS} real historical {PERIOD_DAYS}-day periods."
    ),
    "N_MONTE_CARLO_DRAWS": "Real simulation size -- higher = a smoother real percentile estimate, no change to the underlying real historical distribution being resampled.",
}

INSIGHTS = [
    {
        "headline": "Real Cash-Flow-at-Risk is quantified, not asserted, at every horizon",
        "specific": f"5th-percentile real CFaR at 30/60/90 days: "
                    f"${cfar_results[30]['p5_cfar']:,.0f} / ${cfar_results[60]['p5_cfar']:,.0f} / ${cfar_results[90]['p5_cfar']:,.0f}.",
        "measurable": f"{N_DRAWS:,} real vectorized Monte Carlo draws per horizon, bootstrap-resampled from "
                      f"{real_rates.size} real historical per-period collection rates.",
        "achievable": "Reuses Notebook 01's real period reconstruction directly (HYPER) -- no new data pipeline built.",
        "relevant": "Gives a treasury/ALM function a real, disclosed worst-case cash-inflow band, not a single point estimate.",
        "timebound": "Recompute after each new data refresh; the near-term scheduled-cash assumption should be revisited each time, not left stale.",
    },
    {
        "headline": "The simulation is verified against its own closed-form expectation",
        "specific": f"Monte Carlo sample mean reconciled with the real closed-form expectation at all "
                    f"{len(HORIZONS_DAYS)} horizons, within a {RECONCILIATION_TOLERANCE_PCT:.0%} documented tolerance.",
        "measurable": "Real relative-difference check, printed and logged per horizon (Section 3).",
        "achievable": "Same real cross-check discipline as Mega Project 2 Notebook 03's Monte Carlo vs. closed-form check (Lesson #6).",
        "relevant": "Confirms the vectorized batch simulation has no structural bug before its percentiles are trusted.",
        "timebound": "Re-verified automatically every time this notebook runs.",
    },
]

horizon_table_rows = [
    [f"{h}d", cfar_results[h]["n_periods"], f"{cfar_results[h]['p5_cfar']:,.2f}",
     f"{cfar_results[h]['p50_expected']:,.2f}", f"{cfar_results[h]['p95']:,.2f}",
     f"{cfar_results[h]['relative_diff']:.2%}"]
    for h in HORIZONS_DAYS
]
word_sections = [
    {
        "heading": "Cash-Flow-at-Risk by Horizon",
        "paragraphs": [
            f"Real bootstrap Monte Carlo simulation ({N_DRAWS:,} draws per horizon) resampling "
            f"{real_rates.size} real historical {PERIOD_DAYS}-day period collection rates from "
            f"Notebook 01, applied to an assumed near-term scheduled-cash run-rate of "
            f"${anchor_scheduled_per_period:,.2f} per period.",
        ],
        "table": {
            "headers": ["Horizon", "Real periods", "5th pct (CFaR, $)", "Median expected ($)", "95th pct ($)", "MC vs closed-form diff"],
            "rows": horizon_table_rows,
        },
        "story": [
            f"At the 90-day horizon, the real 5th-percentile outcome (${cfar_results[90]['p5_cfar']:,.0f}) is "
            f"${cfar_results[90]['p50_expected'] - cfar_results[90]['p5_cfar']:,.0f} below the real median "
            f"expected inflow -- the real, quantified downside a treasury function should plan liquidity buffers against.",
        ],
    },
]
word_path = build_word_report(
    REPORTS_DIR / "notebook_02_report.docx",
    title="Mega Project 5 -- Problem 2: Cash-Flow-at-Risk (CFaR) Rolling Forecast",
    subtitle="Home Credit RiskIQ Enterprise Suite -- Liquidity & Cashflow",
    exec_summary=[
        f"Real 90-day median expected cash inflow: ${cfar_results[90]['p50_expected']:,.0f}; "
        f"real 5th-percentile (CFaR): ${cfar_results[90]['p5_cfar']:,.0f}.",
        f"Monte Carlo reconciled with closed-form expectation at all {len(HORIZONS_DAYS)} horizons.",
        f"All {len(checks)} pipeline integrity + statistical checks: {n_pass}/{len(checks)} PASS.",
    ],
    insights=INSIGHTS,
    sections=word_sections,
)

n_draws_ref = assumption_ref(ASSUMPTIONS, "N_MONTE_CARLO_DRAWS")
excel_data_sheets = [
    {"name": "CFaR by Horizon", "headers": ["Horizon (days)", "Real Periods", "5th Pct CFaR", "Median Expected", "95th Pct", "MC vs Closed-Form Diff"],
     "rows": [[h, cfar_results[h]["n_periods"], cfar_results[h]["p5_cfar"], cfar_results[h]["p50_expected"],
                cfar_results[h]["p95"], cfar_results[h]["relative_diff"]] for h in HORIZONS_DAYS]},
    {"name": "Historical Periods", "headers": ["Period start (day)", "Scheduled ($)", "Collected ($)", "Collection Rate"],
     "rows": periods.select(["PERIOD_START_DAY", "SCHEDULED_CASH_AMT", "COLLECTED_CASH_AMT", "DOLLAR_COLLECTION_RATE"]).to_pandas().values.tolist()},
    {"name": "Integrity Checks", "headers": ["Check", "Result"],
     "rows": [[name, "PASS" if ok else "FAIL"] for name, ok in checks]},
]
excel_path = build_excel_workbook(
    REPORTS_DIR / "notebook_02_workbook.xlsx",
    assumptions=ASSUMPTIONS,
    assumption_notes=ASSUMPTION_NOTES,
    data_sheets=excel_data_sheets,
    formula_sheet={
        "name": "CFaR Summary",
        "rows": [
            ("90-Day Median Expected Inflow ($)", cfar_results[90]["p50_expected"]),
            ("90-Day 5th Pct CFaR ($)", cfar_results[90]["p5_cfar"]),
            (f"90-Day Downside vs Median ($, =C1-C2)", f"={cfar_results[90]['p50_expected']}-{cfar_results[90]['p5_cfar']}"),
        ],
    },
    insights_sheet={"name": "Insights & SMART Actions", "items": INSIGHTS},
)

fan_chart = {
    "id": "cfarFanChart", "title": "Real Cash-Flow-at-Risk Fan Chart (5th / 50th / 95th Percentile)", "type": "bar",
    "labels": [f"{h}d" for h in HORIZONS_DAYS],
    "datasets": [
        {"label": "5th pct (CFaR)", "data": [round(cfar_results[h]["p5_cfar"], 2) for h in HORIZONS_DAYS], "backgroundColor": _palette(3)[0]},
        {"label": "Median expected", "data": [round(cfar_results[h]["p50_expected"], 2) for h in HORIZONS_DAYS], "backgroundColor": _palette(3)[1]},
        {"label": "95th pct", "data": [round(cfar_results[h]["p95"], 2) for h in HORIZONS_DAYS], "backgroundColor": _palette(3)[2]},
    ],
    "note": f"Bootstrap Monte Carlo, {N_DRAWS:,} draws per horizon, resampled from {real_rates.size} real historical periods.",
}
history_chart = {
    "id": "historyChart", "title": "Real Historical Dollar Collection Rate by Period (Bootstrap Source)", "type": "line",
    "labels": [str(int(d)) for d in periods["PERIOD_START_DAY"]],
    "datasets": [{"label": "Real $ Collection Rate", "data": [round(float(v), 4) if v is not None else None for v in periods["DOLLAR_COLLECTION_RATE"]],
                  "backgroundColor": _palette(1)[0]}],
    "note": "The exact real historical distribution the Monte Carlo simulation bootstrap-resamples from.",
}

sim_sample = rng.choice(real_rates, size=min(300, N_DRAWS), replace=True)
sample_rows = [[i + 1, round(float(v), 4)] for i, v in enumerate(sim_sample)]

html_path = build_html_dashboard(
    REPORTS_DIR / "notebook_02_dashboard.html",
    title="Mega Project 5 -- Problem 2: Cash-Flow-at-Risk (CFaR) Rolling Forecast",
    subtitle="Real bootstrap Monte Carlo cash-inflow uncertainty bands, 30/60/90-day horizons",
    kpi_cards=[
        {"label": "90d Median Expected", "value": f"${cfar_results[90]['p50_expected']:,.0f}"},
        {"label": "90d CFaR (5th pct)", "value": f"${cfar_results[90]['p5_cfar']:,.0f}"},
        {"label": "Real Historical Periods", "value": f"{real_rates.size}"},
        {"label": "MC Draws / Horizon", "value": f"{N_DRAWS:,}"},
    ],
    charts=[fan_chart, history_chart],
    insights=INSIGHTS,
    data_table={
        "title": "Sampled Real Bootstrap Draws (Historical Rates Resampled)",
        "columns": ["Draw #", "Resampled real historical rate"],
        "rows": sample_rows,
    },
)

csv_written = write_csv_outputs(
    {
        "notebook_02_cfar_by_horizon": __import__("pandas").DataFrame(
            [[h, cfar_results[h]["n_periods"], cfar_results[h]["p5_cfar"], cfar_results[h]["p50_expected"],
              cfar_results[h]["p95"], cfar_results[h]["relative_diff"]] for h in HORIZONS_DAYS],
            columns=["horizon_days", "n_periods", "p5_cfar", "p50_expected", "p95", "mc_vs_closed_form_relative_diff"],
        ),
        "notebook_02_historical_periods": periods.to_pandas(),
    },
    REPORTS_DIR,
)
print(f"[REPORTING] Real reporting package written: {word_path.name}, {excel_path.name}, "
      f"{html_path.name}, plus {len(csv_written)} CSV file(s) (all under decision_engine/reports/).")

# ---------------------------------------------------------------------------
# SECTION 6 — Governance summary JSON (consumed by Notebook 06's Executive Rollup).
# ---------------------------------------------------------------------------
summary = {
    "notebook": "02_cash_flow_at_risk_rolling_forecast",
    "mega_project": 5,
    "problem": 2,
    "n_real_historical_periods": int(real_rates.size),
    "near_term_scheduled_cash_per_period_assumption": anchor_scheduled_per_period,
    "n_monte_carlo_draws": N_DRAWS,
    "cfar_by_horizon": cfar_results,
    "n_checks_total": len(checks),
    "n_checks_pass": n_pass,
    "checks": {name: ok for name, ok in checks},
}
summary_path = REPORTS_DIR / "notebook_02_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

VERDICT = "RECOMMENDED FOR PRODUCTION" if n_pass == len(checks) else "NEEDS REVIEW -- one or more checks FAILED"
print(f"[VERDICT] Deployment readiness: {VERDICT}")
print(f"[DONE] Mega Project 5 / Notebook 02 complete in {time.time() - t0:.1f}s "
      f"using a {PERF['n_threads']}-thread WARP ceiling. {installments.shape[0]:,} real installment rows processed, "
      f"{N_DRAWS * len(HORIZONS_DAYS):,} total Monte Carlo draws.")
